# Module 3 - Simulating Polarisation Dynamics

**Time:** 13:45-15:15  
**Tool focus:** NDlib plus explicit bounded-confidence simulation code


In [ ]:
import matplotlib.pyplot as plt
import ndlib
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import ndlib.models.ModelConfig as mc
import ndlib.models.opinions as op
from ndlib.viz.mpl.OpinionEvolution import OpinionEvolution
from IPython.display import Image, display

sns.set_theme(context="talk", style="whitegrid")


## Simulating Opinion Evolution with Bounded Confidence 

In [ ]:
g = nx.complete_graph(100)

# Model selection
model = op.AlgorithmicBiasModel(g)

# Model configuration
config = mc.Configuration()
config.add_model_parameter("epsilon", 0.32) # bounded confidence
config.add_model_parameter("gamma", 0) # algorithmic bias (none)

model.set_initial_status(config)

# Simulation execution
iterations = model.iteration_bunch(100)

In [ ]:
path = 'plot.png'
OpinionEvolution(model, iterations).plot(filename=str(path))
display(Image(filename=str(path)))

## How Does Social Structure Impact Opinion Evolution?
Let's build a graph with two clear communities

In [ ]:
def build_demo_graph(seed=42, n1=50, n2=50):
    rng = np.random.default_rng(seed)

    sizes = [n1, n2]
    p = [
        [0.18, 0.02],
        [0.02, 0.18],
    ]
    G = nx.stochastic_block_model(sizes, p, seed=seed)
    blocks = G.graph["partition"]
    for community_id, nodes in enumerate(blocks):
        for node in nodes:
            G.nodes[node]["community"] = community_id
    G.add_edge(next(iter(blocks[0])), next(iter(blocks[1])))
    G = nx.convert_node_labels_to_integers(G)

    G.name = None
    return G

def print_mean_variance_by_community(G, opinions, label=""):
    comm_ops = {0: [], 1: []}

    for n in G.nodes():
        comm_ops[G.nodes[n]["community"]].append(opinions[n])

    all_ops = list(opinions.values())

    print(
        f"{label} opinions:\n"
        f"Global: mean={np.mean(all_ops):.3f}, var={np.var(all_ops):.3f}\n"
        f"C0: mean={np.mean(comm_ops[0]):.3f}, var={np.var(comm_ops[0]):.3f}\n"
        f"C1: mean={np.mean(comm_ops[1]):.3f}, var={np.var(comm_ops[1]):.3f}"
    )

In [ ]:
G = build_demo_graph()
print(G)

In [ ]:
# Model selection
model = op.AlgorithmicBiasModel(G)

# Model configuration
config = mc.Configuration()
config.add_model_parameter("epsilon", 0.32) # bounded confidence
config.add_model_parameter("gamma", 0) # algorithmic bias (none)

model.set_initial_status(config)


In [ ]:
print_mean_variance_by_community(G, model.status, "Initial")

In [ ]:
# Simulation execution
iterations = model.iteration_bunch(100)

In [ ]:
print_mean_variance_by_community(G, model.status, "Final")

In [ ]:
path = 'plot.png'
OpinionEvolution(model, iterations).plot(filename=str(path))
display(Image(filename=str(path)))

## How Does Opinion Distribution Impact Opinion Evolution?
Let's edit initial opinions so that one community is polarized, while the other has bell-shaped opinion distribution.

In [ ]:
rng = np.random.default_rng(42)
custom_opinions = {}

for node in G.nodes:
    community = G.nodes[node]["community"]

    if community == 0: # bell-shaped 
        custom_opinions[node] = float(np.clip(rng.normal(0.45, 0.12), 0, 1))
    else:
        custom_opinions[node] = 1.0 # polarized


In [ ]:
# Model selection
model = op.AlgorithmicBiasModel(G)

# Model configuration
config = mc.Configuration()
config.add_model_parameter("epsilon", 0.32) # bounded confidence
config.add_model_parameter("gamma", 0) # algorithmic bias (none)

model.set_initial_status(config)
model.status = custom_opinions.copy()
model.initial_status = custom_opinions.copy()



In [ ]:
print_mean_variance_by_community(G, model.status, "Initial")

In [ ]:
# Simulation execution
iterations = model.iteration_bunch(100)

In [ ]:
print_mean_variance_by_community(G, model.status, "Final")

In [ ]:
path = 'plot.png'
OpinionEvolution(model, iterations).plot(filename=str(path))
display(Image(filename=str(path)))